# MRI-to-PET GAN Model (Memory Optimized)
## Lightweight Pix2Pix for Limited GPU Resources

A memory-efficient conditional GAN for synthesizing PET images from MRI.

**Optimizations:**
- Lightweight U-Net Generator (32 base channels)
- Simple PatchGAN Discriminator
- No VGG perceptual loss (saves ~500MB)
- Gradient checkpointing for memory savings
- Batch size = 1
- Mixed precision (FP16)

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os, gc, math, copy, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torch.utils.checkpoint import checkpoint

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as compare_ssim
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from scipy.ndimage import rotate as scipy_rotate
from tqdm.notebook import tqdm

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    # Memory optimization
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
# ============================================================
# CONFIGURATION (Memory Optimized)
# ============================================================

class Config:
    # Paths
    DATA_ROOT = r"C:/Users/kdesh/OneDrive/Documents/Year 4/Implementation"
    SAVE_DIR  = r"C:/Users/kdesh/OneDrive/Documents/Year 4/Implementation/checkpoints_gan"

    # Data
    IMG_H, IMG_W = 160, 192
    MIN_BRAIN_PX = 800
    TRAIN_SPLIT  = 0.80
    VAL_SPLIT    = 0.10

    # Generator (Lightweight)
    G_CHANNELS = [32, 64, 128, 256]  # Much smaller than standard
    G_USE_CHECKPOINT = True          # Gradient checkpointing

    # Discriminator (Lightweight)
    D_CHANNELS = [32, 64, 128]

    # Loss weights (no VGG)
    LAMBDA_ADV  = 1.0
    LAMBDA_L1   = 100.0
    LAMBDA_SSIM = 10.0

    # Training (Conservative)
    EPOCHS       = 100
    BATCH_SIZE   = 1               # Minimum batch size
    G_LR         = 2e-4
    D_LR         = 2e-4
    GRAD_CLIP    = 1.0
    EMA_DECAY    = 0.999

    # Augmentation (lightweight)
    AUG_FLIP     = 0.5
    AUG_ROTATE   = 0.3

    # Checkpointing
    SAVE_EVERY   = 5
    SAMPLE_EVERY = 5
    NUM_WORKERS  = 0               # Reduce memory overhead

cfg = Config()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)
print(f"Save dir: {cfg.SAVE_DIR}")

In [ ]:
# ============================================================
# DATASET (Simplified)
# ============================================================

class MRItoPETDataset(Dataset):
    """Memory-efficient 2.5D MRI-to-PET dataset."""

    def __init__(self, data_root, patient_ids, cfg, augment=False):
        self.data_root = data_root
        self.cfg = cfg
        self.augment = augment
        self.samples = []

        for pid in patient_ids:
            mri_path = os.path.join(data_root, pid, 'mri_processed.npy')
            pet_path = os.path.join(data_root, pid, 'pet_processed.npy')

            if not (os.path.exists(mri_path) and os.path.exists(pet_path)):
                continue

            try:
                mri = np.load(mri_path, mmap_mode='r')  # Memory-mapped
                pet = np.load(pet_path, mmap_mode='r')
                if mri.shape != pet.shape:
                    continue
                D = mri.shape[2]
                for k in range(1, D - 1):
                    if np.sum(np.abs(pet[:, :, k]) > 0.1) >= cfg.MIN_BRAIN_PX:
                        self.samples.append((mri_path, pet_path, k))
            except:
                continue

        print(f"Dataset: {len(patient_ids)} patients, {len(self.samples)} slices")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        mri_path, pet_path, k = self.samples[idx]
        
        # Load only needed slices (memory efficient)
        mri = np.load(mri_path, mmap_mode='r')
        pet = np.load(pet_path, mmap_mode='r')
        
        mri_stack = np.stack([mri[:, :, k-1], mri[:, :, k], mri[:, :, k+1]], axis=0).astype(np.float32)
        pet_slice = pet[:, :, k].astype(np.float32)

        # Simple augmentation
        if self.augment:
            if random.random() < self.cfg.AUG_FLIP:
                mri_stack = mri_stack[:, :, ::-1].copy()
                pet_slice = pet_slice[:, ::-1].copy()
            if random.random() < self.cfg.AUG_ROTATE:
                angle = random.uniform(-5, 5)
                for i in range(3):
                    mri_stack[i] = scipy_rotate(mri_stack[i], angle, reshape=False, order=1)
                pet_slice = scipy_rotate(pet_slice, angle, reshape=False, order=1)

        return torch.from_numpy(mri_stack.copy()), torch.from_numpy(pet_slice[None].copy())


# Create datasets
all_patients = sorted([d for d in os.listdir(cfg.DATA_ROOT) 
                       if os.path.isdir(os.path.join(cfg.DATA_ROOT, d)) and d.startswith('NACC')])
random.shuffle(all_patients)

n_train = int(len(all_patients) * cfg.TRAIN_SPLIT)
n_val = int(len(all_patients) * cfg.VAL_SPLIT)

train_patients = all_patients[:n_train]
val_patients = all_patients[n_train:n_train + n_val]
test_patients = all_patients[n_train + n_val:]

print(f"Patients: Train={len(train_patients)}, Val={len(val_patients)}, Test={len(test_patients)}")

train_dataset = MRItoPETDataset(cfg.DATA_ROOT, train_patients, cfg, augment=True)
val_dataset = MRItoPETDataset(cfg.DATA_ROOT, val_patients, cfg, augment=False)

train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, 
                          num_workers=cfg.NUM_WORKERS, pin_memory=False)
val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False,
                        num_workers=cfg.NUM_WORKERS, pin_memory=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# ============================================================
# LIGHTWEIGHT GENERATOR (U-Net)
# ============================================================

class ConvBlock(nn.Module):
    """Simple conv block: Conv -> BatchNorm -> LeakyReLU."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
        )
    
    def forward(self, x):
        return self.conv(x)


class Generator(nn.Module):
    """Lightweight U-Net Generator with optional gradient checkpointing."""
    
    def __init__(self, in_ch=3, out_ch=1, channels=[32, 64, 128, 256], use_checkpoint=False):
        super().__init__()
        self.use_checkpoint = use_checkpoint
        
        # Encoder
        self.enc1 = ConvBlock(in_ch, channels[0])
        self.enc2 = ConvBlock(channels[0], channels[1])
        self.enc3 = ConvBlock(channels[1], channels[2])
        self.enc4 = ConvBlock(channels[2], channels[3])
        
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = ConvBlock(channels[3], channels[3])
        
        # Decoder
        self.up4 = nn.ConvTranspose2d(channels[3], channels[3], 2, stride=2)
        self.dec4 = ConvBlock(channels[3] * 2, channels[2])
        
        self.up3 = nn.ConvTranspose2d(channels[2], channels[2], 2, stride=2)
        self.dec3 = ConvBlock(channels[2] * 2, channels[1])
        
        self.up2 = nn.ConvTranspose2d(channels[1], channels[1], 2, stride=2)
        self.dec2 = ConvBlock(channels[1] * 2, channels[0])
        
        self.up1 = nn.ConvTranspose2d(channels[0], channels[0], 2, stride=2)
        self.dec1 = ConvBlock(channels[0] * 2, channels[0])
        
        # Output
        self.out = nn.Conv2d(channels[0], out_ch, 1)
    
    def _enc_block(self, block, x):
        if self.use_checkpoint and self.training:
            return checkpoint(block, x, use_reentrant=False)
        return block(x)
    
    def forward(self, x):
        # Encoder
        e1 = self._enc_block(self.enc1, x)
        e2 = self._enc_block(self.enc2, self.pool(e1))
        e3 = self._enc_block(self.enc3, self.pool(e2))
        e4 = self._enc_block(self.enc4, self.pool(e3))
        
        # Bottleneck
        b = self._enc_block(self.bottleneck, self.pool(e4))
        
        # Decoder with skip connections
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        
        return torch.tanh(self.out(d1))

In [ ]:
# ============================================================
# LIGHTWEIGHT DISCRIMINATOR (PatchGAN)
# ============================================================

class Discriminator(nn.Module):
    """Simple PatchGAN discriminator."""
    
    def __init__(self, in_ch=4, channels=[32, 64, 128]):
        super().__init__()
        
        layers = []
        prev_ch = in_ch
        
        for i, ch in enumerate(channels):
            layers.append(nn.Conv2d(prev_ch, ch, 4, stride=2, padding=1))
            if i > 0:
                layers.append(nn.BatchNorm2d(ch))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            prev_ch = ch
        
        layers.append(nn.Conv2d(prev_ch, 1, 4, padding=1))
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, mri, pet):
        x = torch.cat([mri, pet], dim=1)
        return self.model(x)

In [ ]:
# ============================================================
# LOSS FUNCTIONS (Simplified - No VGG)
# ============================================================

class SSIMLoss(nn.Module):
    """Simple SSIM loss."""
    def __init__(self, window_size=11):
        super().__init__()
        self.window_size = window_size
        self.C1 = 0.01 ** 2
        self.C2 = 0.03 ** 2

    def forward(self, pred, target):
        mu1 = F.avg_pool2d(pred, self.window_size, stride=1, padding=self.window_size//2)
        mu2 = F.avg_pool2d(target, self.window_size, stride=1, padding=self.window_size//2)
        
        mu1_sq, mu2_sq = mu1 ** 2, mu2 ** 2
        mu1_mu2 = mu1 * mu2
        
        sigma1_sq = F.avg_pool2d(pred ** 2, self.window_size, stride=1, padding=self.window_size//2) - mu1_sq
        sigma2_sq = F.avg_pool2d(target ** 2, self.window_size, stride=1, padding=self.window_size//2) - mu2_sq
        sigma12 = F.avg_pool2d(pred * target, self.window_size, stride=1, padding=self.window_size//2) - mu1_mu2
        
        ssim = ((2 * mu1_mu2 + self.C1) * (2 * sigma12 + self.C2)) / \
               ((mu1_sq + mu2_sq + self.C1) * (sigma1_sq + sigma2_sq + self.C2))
        
        return 1 - ssim.mean()


def d_loss_fn(real_out, fake_out):
    """LSGAN discriminator loss."""
    return 0.5 * (torch.mean((real_out - 1) ** 2) + torch.mean(fake_out ** 2))


def g_loss_fn(fake_out):
    """LSGAN generator loss."""
    return torch.mean((fake_out - 1) ** 2)

In [ ]:
# ============================================================
# EMA (Exponential Moving Average)
# ============================================================

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.model = copy.deepcopy(model)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for ema_p, p in zip(self.model.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(p.data, alpha=1 - self.decay)

In [ ]:
# ============================================================
# TRAINING LOOP (Memory Optimized)
# ============================================================

@torch.no_grad()
def evaluate(G, loader, device):
    """Quick evaluation."""
    G.eval()
    ssims = []
    
    for mri, pet in loader:
        mri, pet = mri.to(device), pet.to(device)
        fake = G(mri)
        
        for i in range(fake.shape[0]):
            f = fake[i, 0].cpu().numpy()
            r = pet[i, 0].cpu().numpy()
            ssims.append(compare_ssim(f, r, data_range=2.0))
    
    return np.mean(ssims) if ssims else 0.0


def train(cfg, train_loader, val_loader):
    # Clear memory
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    # Build models
    G = Generator(in_ch=3, out_ch=1, channels=cfg.G_CHANNELS, 
                  use_checkpoint=cfg.G_USE_CHECKPOINT).to(device)
    D = Discriminator(in_ch=4, channels=cfg.D_CHANNELS).to(device)
    
    # Print model sizes
    g_params = sum(p.numel() for p in G.parameters()) / 1e6
    d_params = sum(p.numel() for p in D.parameters()) / 1e6
    print(f"Generator: {g_params:.2f}M params")
    print(f"Discriminator: {d_params:.2f}M params")
    
    # Optimizers
    opt_G = torch.optim.Adam(G.parameters(), lr=cfg.G_LR, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=cfg.D_LR, betas=(0.5, 0.999))
    
    # EMA
    ema = EMA(G, decay=cfg.EMA_DECAY)
    
    # Loss
    ssim_loss = SSIMLoss().to(device)
    
    # Mixed precision
    scaler = GradScaler('cuda') if device.type == 'cuda' else None
    
    # Training state
    best_ssim = 0.0
    history = {'g_loss': [], 'd_loss': [], 'ssim': []}
    
    print(f"\nStarting training for {cfg.EPOCHS} epochs...")
    
    for epoch in range(1, cfg.EPOCHS + 1):
        G.train()
        D.train()
        g_losses, d_losses = [], []
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.EPOCHS}", leave=False)
        for mri, pet in pbar:
            mri = mri.to(device)
            pet = pet.to(device)
            
            # ---- Train Discriminator ----
            opt_D.zero_grad(set_to_none=True)
            
            with autocast('cuda', enabled=scaler is not None):
                with torch.no_grad():
                    fake = G(mri)
                real_out = D(mri, pet)
                fake_out = D(mri, fake.detach())
                loss_D = d_loss_fn(real_out, fake_out)
            
            if scaler:
                scaler.scale(loss_D).backward()
                scaler.step(opt_D)
            else:
                loss_D.backward()
                opt_D.step()
            
            # ---- Train Generator ----
            opt_G.zero_grad(set_to_none=True)
            
            with autocast('cuda', enabled=scaler is not None):
                fake = G(mri)
                fake_out = D(mri, fake)
                
                loss_adv = g_loss_fn(fake_out)
                loss_l1 = F.l1_loss(fake, pet)
                loss_ssim = ssim_loss(fake, pet)
                
                loss_G = cfg.LAMBDA_ADV * loss_adv + cfg.LAMBDA_L1 * loss_l1 + cfg.LAMBDA_SSIM * loss_ssim
            
            if scaler:
                scaler.scale(loss_G).backward()
                scaler.unscale_(opt_G)
                torch.nn.utils.clip_grad_norm_(G.parameters(), cfg.GRAD_CLIP)
                scaler.step(opt_G)
                scaler.update()
            else:
                loss_G.backward()
                torch.nn.utils.clip_grad_norm_(G.parameters(), cfg.GRAD_CLIP)
                opt_G.step()
            
            # Update EMA
            ema.update(G)
            
            g_losses.append(loss_G.item())
            d_losses.append(loss_D.item())
            pbar.set_postfix({'G': f'{loss_G.item():.3f}', 'D': f'{loss_D.item():.3f}'})
        
        # Epoch stats
        avg_g = np.mean(g_losses)
        avg_d = np.mean(d_losses)
        history['g_loss'].append(avg_g)
        history['d_loss'].append(avg_d)
        
        # Validation
        if epoch % cfg.SAMPLE_EVERY == 0 or epoch == cfg.EPOCHS:
            val_ssim = evaluate(ema.model, val_loader, device)
            history['ssim'].append((epoch, val_ssim))
            print(f"Epoch {epoch}: G={avg_g:.4f}, D={avg_d:.4f}, SSIM={val_ssim:.4f}")
            
            # Save best
            if val_ssim > best_ssim:
                best_ssim = val_ssim
                torch.save({
                    'epoch': epoch,
                    'G': G.state_dict(),
                    'ema': ema.model.state_dict(),
                    'ssim': val_ssim,
                }, os.path.join(cfg.SAVE_DIR, 'best_model.pt'))
                print(f"  -> Saved best model (SSIM={val_ssim:.4f})")
        else:
            print(f"Epoch {epoch}: G={avg_g:.4f}, D={avg_d:.4f}")
        
        # Checkpoint
        if epoch % cfg.SAVE_EVERY == 0:
            torch.save({
                'epoch': epoch,
                'G': G.state_dict(),
                'D': D.state_dict(),
                'ema': ema.model.state_dict(),
                'opt_G': opt_G.state_dict(),
                'opt_D': opt_D.state_dict(),
            }, os.path.join(cfg.SAVE_DIR, 'checkpoint.pt'))
        
        # Memory cleanup
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    
    print(f"\nTraining complete! Best SSIM: {best_ssim:.4f}")
    return G, ema, history


# Run training
G, ema, history = train(cfg, train_loader, val_loader)

In [ ]:
# ============================================================
# VISUALIZATION
# ============================================================

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['g_loss'], label='Generator')
axes[0].plot(history['d_loss'], label='Discriminator')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_title('Training Loss')

if history['ssim']:
    epochs, ssims = zip(*history['ssim'])
    axes[1].plot(epochs, ssims, 'o-')
    axes[1].axhline(y=0.9, color='r', linestyle='--', label='Target')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('SSIM')
    axes[1].legend()
    axes[1].set_title('Validation SSIM')

plt.tight_layout()
plt.savefig(os.path.join(cfg.SAVE_DIR, 'training_curves.png'), dpi=100)
plt.show()

In [ ]:
# ============================================================
# GENERATE SAMPLES
# ============================================================

@torch.no_grad()
def show_samples(model, loader, n=4):
    model.eval()
    fig, axes = plt.subplots(3, n, figsize=(3*n, 9))
    
    samples = 0
    for mri, pet in loader:
        mri, pet = mri.to(device), pet.to(device)
        fake = model(mri)
        
        for i in range(mri.shape[0]):
            if samples >= n:
                break
            
            axes[0, samples].imshow(mri[i, 1].cpu(), cmap='gray', vmin=-1, vmax=1)
            axes[0, samples].set_title('MRI')
            axes[0, samples].axis('off')
            
            axes[1, samples].imshow(pet[i, 0].cpu(), cmap='hot', vmin=-1, vmax=1)
            axes[1, samples].set_title('Real PET')
            axes[1, samples].axis('off')
            
            axes[2, samples].imshow(fake[i, 0].cpu(), cmap='hot', vmin=-1, vmax=1)
            ssim = compare_ssim(fake[i,0].cpu().numpy(), pet[i,0].cpu().numpy(), data_range=2.0)
            axes[2, samples].set_title(f'Fake (SSIM={ssim:.3f})')
            axes[2, samples].axis('off')
            
            samples += 1
        
        if samples >= n:
            break
    
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.SAVE_DIR, 'samples.png'), dpi=150)
    plt.show()

# Load best model and show samples
ckpt = torch.load(os.path.join(cfg.SAVE_DIR, 'best_model.pt'), map_location=device)
ema.model.load_state_dict(ckpt['ema'])
print(f"Loaded best model from epoch {ckpt['epoch']} (SSIM={ckpt['ssim']:.4f})")

show_samples(ema.model, val_loader, n=4)